# Notebook 06: Quark Propagators

**Learning objectives:**
- Create a point source for the quark propagator
- Solve $D \cdot S = \delta$ for the propagator
- Visualize the propagator magnitude vs distance
- Understand the 8 propagator components (2 colors $\times$ 4 spins)

**Prerequisites:** Notebook 05

**From modular code:** `MesonBase.create_point_source()`, `MesonBase.solve_dirac_system()`

In [ ]:
from notebook_utils import setup_paths, load_config
REPO = setup_paths()

import os
import numpy as np
import matplotlib.pyplot as plt
import su2
from MesonBase import (build_wilson_dirac_matrix,
                       generate_identity_gauge_field,
                       create_point_source,
                       solve_dirac_system)

## 1. Point Sources

A point source $\delta^{(3)}(\vec{x})\,\delta_{c,c_0}\,\delta_{\alpha,\alpha_0}$
creates a quark at the origin with definite color $c_0$ and spin $\alpha_0$.

The **quark propagator** $S(x, 0) = D_W^{-1}(x, 0)$ describes how a quark
created at the origin propagates to point $x$. It satisfies:
$$D_W \cdot S = \delta \qquad \Leftrightarrow \qquad S = D_W^{-1}\,\delta$$

For SU(2) with $N_c = 2$ colors and $N_s = 4$ spins, we need
$N_c \times N_s = 8$ inversions (one per color-spin source combination)
to build the full propagator matrix.

In [ ]:
La = [4, 4, 4, 4]
V = su2.vol(La)

# Create a point source: color=0, spin=0, time-slice=0
source = create_point_source(La, t_source=0, color=0, spin=0, verbose=True)

print(f"Source vector length: {len(source)}")
print(f"Non-zero entries: {np.count_nonzero(source)}")
print(f"Non-zero at index: {np.nonzero(source)[0][0]}")

## 2. Solving the Dirac Equation

We solve $D_W \cdot S = \delta$ where $D_W$ is the Wilson-Dirac operator
and $\delta$ is the point source. This is the most expensive part of
lattice QCD calculations.

On our $4^4$ lattice this is a $2048 \times 2048$ linear system.

In [ ]:
# TRY: Change mass from 0.1 to 0.5 — heavier quarks decay faster with distance
# TRY: Change wilson_r from 0.5 to 1.0 — affects the effective mass (m + 4r)
mass = 0.1
wilson_r = 0.5

# Use identity gauge field (free field)
U_free, _ = generate_identity_gauge_field(La)
D = build_wilson_dirac_matrix(mass, La, wilson_r=wilson_r, U=U_free,
                              verbose=True)

# Solve for one source
prop = solve_dirac_system(D, source, method='auto', verbose=True)
print(f"\nPropagator norm: {np.linalg.norm(prop):.6f}")

## 3. All 8 Propagators

To compute meson correlators, we need the propagator from all
8 source types (2 colors $\times$ 4 spins).

In [ ]:
propagators = []
for color in range(2):
    for spin in range(4):
        src = create_point_source(La, t_source=0, color=color, spin=spin)
        sol = solve_dirac_system(D, src, method='auto')
        propagators.append(sol)
        print(f"  color={color}, spin={spin}: norm = {np.linalg.norm(sol):.4f}")

print(f"\nTotal propagators: {len(propagators)}")

## 4. Propagator vs Distance

The quark propagator should fall off exponentially with distance:
$|S(x, 0)| \sim e^{-m_{\rm eff} |x|}$.

Let's visualize the magnitude along the time direction.

In [ ]:
# Sum propagator magnitude over spatial volume at each time slice
Lx, Ly, Lz, Lt = La
prop_vs_t = np.zeros(Lt)

# Use the first propagator (color=0, spin=0)
prop0 = propagators[0]

for t in range(Lt):
    for x in range(Lx):
        for y in range(Ly):
            for z in range(Lz):
                idx = su2.p2i(np.array([x, y, z, t]), La)
                base = 8 * idx
                # Sum over all color-spin components at this site
                for cs in range(8):
                    prop_vs_t[t] += abs(prop0[base + cs])**2

plt.figure(figsize=(7, 4))
plt.semilogy(range(Lt), prop_vs_t, 'o-', markersize=8)
plt.xlabel('Time slice t')
plt.ylabel(r'$\sum_{\vec{x}} |S(\vec{x},t; 0)|^2$')
plt.title('Quark propagator magnitude vs time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Propagator on a Gauge Background

Now let's solve on a non-trivial (interacting) gauge field.

In [ ]:
# Load a sample configuration
config_path = os.path.join(REPO, "configs", "sample_4x4x4x4",
                           "random_4x4x4x4.pkl")
U_rand, meta = load_config(config_path)
print(f"Loaded config, plaquette: {meta.get('plaquette', 'N/A')}")

U_interacting = [None, U_rand]  # format expected by build_wilson_dirac_matrix
D_int = build_wilson_dirac_matrix(mass, La, wilson_r=wilson_r,
                                  U=U_interacting, verbose=True)

# Solve for one source
prop_int = solve_dirac_system(D_int, source, verbose=True)

# Compare norms
print(f"\nFree field propagator norm:     {np.linalg.norm(prop):.4f}")
print(f"Interacting propagator norm: {np.linalg.norm(prop_int):.4f}")

The gauge field modifies the propagator -- quarks interact with the
gluon background, which is the origin of confinement and hadron masses.

## Exercises

1. Compare the propagator magnitude vs time for free-field and interacting cases.
   How does the gauge field affect the decay rate?
2. Change the quark mass to 0.5 and re-solve. How does the propagator change?
3. Try different solver methods (`'direct'`, `'gmres'`) and compare
   the solution norms and residuals.
4. **All 8 propagators**: Using the code in Section 3, examine the
   $8 \times 8$ propagator matrix at the source site ($t=0$, origin).
   Reshape: for each source $(c_s, s_s)$ and sink $(c_k, s_k)$, extract
   `propagators[4*c_s + s_s][8*source_idx + 2*s_k + c_k]`.
   Which components are largest? The matrix should be approximately
   block-diagonal in color — verify this.
5. **Solver comparison**: Time `'direct'` vs `'gmres'` for lattice sizes
   $2^4$ and $4^4$. Use `import time; t0 = time.time()`.
   At what volume does iterative solving become faster than direct?
   Plot solve time vs lattice volume.

## Exercises

1. Compare the propagator magnitude vs time for free-field and interacting cases.
   How does the gauge field affect the decay rate?
2. Change the quark mass to 0.5 and re-solve. How does the propagator change?
3. Try different solver methods (`'direct'`, `'gmres'`) and compare
   the solution norms and residuals.